# avgpool-reduce — ex2: build global avgpool via einops.reduce

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `avgpool-reduce`. Running the final beacon cell reports progress against the `CNN: AvgPool as reduce` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: AvgPool as reduce` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`avgpool-reduce`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "avgpool-reduce"
DD_SUBTOPIC = "CNN: AvgPool as reduce"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Global average pooling — quick refresher

Global avg-pool collapses the ENTIRE spatial extent into one scalar per (batch, channel). The einops form is the cleanest:

```
y = einops.reduce(x, 'b c h w -> b c', 'mean')
```

Input `(B, C, H, W)` → output `(B, C)`. The `h` and `w` axes are absent from the right side — that means they're reduced (averaged out).

**Why ResNet uses this.** The last conv block of ResNet outputs `(B, 512, 7, 7)` (for 224×224 input). Before the classifier `Linear(512, num_classes)` can consume it, the spatial dims must collapse. Global avg-pool is the canonical way:

```
(B, 512, 7, 7)  →  mean over (h, w)  →  (B, 512)  →  Linear  →  (B, num_classes)
```

**Equivalence.** `nn.AdaptiveAvgPool2d((1, 1))(x).squeeze(-1).squeeze(-1)` produces an identical tensor. The einops form is more transparent — you SEE the reduction in the pattern string.

**Contrast with local AvgPool2d.** Local pool keeps spatial structure (just at lower resolution); global pool destroys spatial structure entirely. Both use the same `mean` reduction — only the pattern string (factor pattern vs. straight reduce) differs.

### Exercise 2 — build global avgpool via einops.reduce

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `einops.reduce` with a full spatial collapse pattern to implement global average pooling — turning a `(B, C, H, W)` feature map into a `(B, C)` tensor — and verify against `nn.AdaptiveAvgPool2d`.
> Keywords: avgpool, global-pool, einops-reduce, resnet-head
> ```

**KCs targeted:** `global-avgpool-collapse`, `adaptive-pool-equivalence`

Implement `ex2_global_avgpool(x)`. Given input `x: (B, C, H, W)`, return a `(B, C)` tensor whose entries are the **mean** of each channel's entire spatial extent.

**Use einops.reduce with a spatial-collapse pattern.**

```
einops.reduce(x, 'b c h w -> b c', 'mean')
```

The `h` and `w` axes are ABSENT from the right side — meaning they're reduced away. Compared to the LOCAL avgpool pattern `'b c (h p1) (w p2) -> b c h w'`, here there's no factoring because we want to collapse the ENTIRE spatial extent, not block-by-block.

**Why this is its own atom.** ResNet's classifier head is literally `global_avgpool → Linear → loss`. Turning `(B, 512, 7, 7)` into `(B, 512)` is the bottleneck — once you have it, the linear layer can produce class logits.

**Shape contract.** Input `(B, C, H, W)`; output `(B, C)`. The test deliberately uses non-square `(H, W)` to make sure you're not accidentally assuming `H == W`.

The test cross-checks against `nn.AdaptiveAvgPool2d((1, 1))(x).squeeze(-1).squeeze(-1)` and confirms identical output.

In [ ]:
def ex2_global_avgpool(x: Tensor) -> Tensor:
    return einops.reduce(x, 'b c h w -> b c', 'mean')


<details><summary>Solution</summary>

```python
def ex2_global_avgpool(x: Tensor) -> Tensor:
    return einops.reduce(x, 'b c h w -> b c', 'mean')
```

**Why the einops pattern is so terse.** `'b c h w -> b c'` says: 'keep b and c, drop h and w'. The `'mean'` argument says how to drop them. One line, no kernel-size or stride to compute — and it generalizes to N-D pooling by adding more axes.

**Equivalent rewrites.**
- `x.mean(dim=(-2, -1))` — same result, less self-documenting.
- `nn.AdaptiveAvgPool2d((1, 1))(x).squeeze(-1).squeeze(-1)` — PyTorch's idiom; works but has the awkward `.squeeze(-1)` boilerplate to remove the size-1 spatial dims.
- `einops.reduce(x, 'b c h w -> b c ()', 'mean').squeeze(-1)` — intermediate form; uses anonymous-axis syntax `()` to keep spatial dims as size-1, then squeezes. The bare pattern (no spatial axes in output) is cleaner.

**Why ResNet uses global avgpool.** It collapses spatial extent into a single per-channel summary statistic, which makes the model **invariant to spatial position** of features. A cat in the top-left and a cat in the bottom-right produce the same global-pool output (modulo translation-equivariance of the conv layers above). This is a desirable inductive bias for classification.

**Contrast with flatten + Linear.** The pre-ResNet pattern was `flatten(x)` → `Linear(C*H*W, num_classes)`. For a 512-channel 7x7 input, that's `25088 * 1000 ~= 25M` params just for the head. Global pool reduces this to `512 * 1000 = 512K` — 50× smaller and immune to spatial reshuffling.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()